In [1]:
!pip install numpy
!pip install pandas
!pip install tensorflow
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os 
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
train_dir = r'D:\archive\chest_xray\train/'
categories = ['PNEUMONIA', 'NORMAL']
filepaths = []
labels = []

In [4]:
for category in categories:
    folder = os.path.join(train_dir, category)
    for fname in os.listdir(folder):
        filepaths.append(f"{category}/{fname}")
        labels.append(category)

In [5]:
df = pd.DataFrame({'Filename' : filepaths, 'Label' : labels})

In [6]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

In [7]:
train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=train_dir,
    x_col='Filename',
    y_col='Label',
    subsets='training',
    batch_size=32,
    seed=42,
    shuffles=True,
    class_mode='binary',
    target_size=(150,150)
)

Found 5216 validated image filenames belonging to 2 classes.


In [8]:
valid_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=train_dir,
    x_col='Filename',
    y_col='Label',
    subsets='validation',
    batch_size=32,
    seed=42,
    shuffles=True,
    class_mode='binary',
    target_size=(150,150)
)

Found 5216 validated image filenames belonging to 2 classes.


In [9]:
model = Sequential([
    Conv2D(32,(3,3),activation='relu', input_shape=(150,150,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(3,3),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(3,3),
    Flatten(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

C:\Users\miana\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 24, 24, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 22, 22, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 7, 7, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 6272)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 6272)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         802,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 896,321 (3.42 MB)

 Trainable params: 896,321 (3.42 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
!pip install pillow

Defaulting to user installation because normal site-packages is not writeable


In [12]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=3,
    validation_data=valid_generator,
    validation_steps=valid_generator.samples // train_generator.batch_size
)

Epoch 1/3
163/163 ━━━━━━━━━━━━━━━━━━━━ 370s 2s/step - accuracy: 0.8873 - loss: 0.2637 - val_accuracy: 0.9536 - val_loss: 0.1257
Epoch 2/3
163/163 ━━━━━━━━━━━━━━━━━━━━ 284s 2s/step - accuracy: 0.9544 - loss: 0.1267 - val_accuracy: 0.9664 - val_loss: 0.0886
Epoch 3/3
163/163 ━━━━━━━━━━━━━━━━━━━━ 264s 2s/step - accuracy: 0.9653 - loss: 0.1018 - val_accuracy: 0.9781 - val_loss: 0.0652


In [13]:
test_dir = r'D:\archive\chest_xray\test/'
test_filepaths = []
test_labels = []

In [14]:
for category in categories:
    folder = os.path.join(test_dir, category)
    for fname in os.listdir(folder):
        test_filepaths.append(f"{category}/{fname}")
        test_labels.append(category)

In [15]:
test_df = pd.DataFrame({'Filename' : test_filepaths, 'Label' : test_labels})

In [16]:
test_datagen = ImageDataGenerator(rescale=1./255)

In [17]:
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=test_dir,
    x_col='Filename',
    y_col='Label',
    batch_size=32,
    shuffles=False,
    class_mode='binary',
    target_size=(150,150)
)

Found 624 validated image filenames belonging to 2 classes.


In [18]:
pred_probs = model.predict(test_generator)
preds = (pred_probs > 0.5).astype(int).flatten()
true_labels = test_generator.classes
class_labels = list(test_generator.class_indices.keys())
print(classification_report(true_labels, preds, target_names=class_labels))
print(confusion_matrix(true_labels, preds))

20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 905ms/step
              precision    recall  f1-score   support

      NORMAL       0.44      0.18      0.25       234
   PNEUMONIA       0.64      0.86      0.73       390

    accuracy                           0.61       624
   macro avg       0.54      0.52      0.49       624
weighted avg       0.56      0.61      0.55       624

[[ 42 192]
 [ 54 336]]
